# logits 的完整生成流程

## 场景设定
假设模型已经生成了 "今晚我想吃"，现在要预测下一个词。

词汇表只有 5 个词：

|  ID | 词  |
| :-: | :- |
|  0  | 火锅 |
|  1  | 烧烤 |
|  2  | 沙拉 |
|  3  | 面条 |
|  4  | 石头 |

## 第一步：模型输出 logits
模型内部有一个线性层，把隐藏状态映射到词汇表维度，输出 5 个原始分数：


In [ ]:
logits = [3.2, 2.8, -1.5, 1.0, -10.0]
         ↑火锅  ↑烧烤  ↑沙拉  ↑面条  ↑石头

### 关键特性
- **没有上下界**：可以是正数、负数、很大或很小
- **不是概率**：不能直接当百分比用
- **是相对分数**：分数越高，模型越"看好"这个词


## 第二步：变成概率（Softmax）
<font color='red'>用 softmax 把 logits 转换成概率分布：</font>

### 注意：
 - **softmax 会放大差距**——火锅和烧烤分数只差 0.4，但概率差了 16 个百分点；石头分数极低，几乎被"抹除"。

In [ ]:
P(火锅)   = e^3.2  / Σ ≈ 24.5 / 48.6 ≈ 50.4%
P(烧烤)   = e^2.8  / Σ ≈ 16.4 / 48.6 ≈ 33.8%
P(沙拉)   = e^-1.5 / Σ ≈ 0.22 / 48.6 ≈ 0.5%
P(面条)   = e^1.0  / Σ ≈ 2.72 / 48.6 ≈ 5.6%
P(石头)   = e^-10  / Σ ≈ 0.000045 / 48.6 ≈ 0.00009%

## 第三步：根据策略选词

| 策略              | 做法               | 结果                    |
| :-------------- | :--------------- | :-------------------- |
| **贪心 (Greedy)** | 直接选概率最高的         | **火锅**                |
| **温度采样**        | <font color='red'>给 logits 除以温度 T </font> | T=2 时更随机，T=0.5 时更确定   |
| **Top-k**       | **只从概率最高的 k 个里选**    | k=2 时只在火锅/烧烤里挑        |
| **Top-p**       | **从累积概率达 p 的最小集合里选** | p=0.9 时选 {火锅, 烧烤, 面条} |


## 直观类比
想象 logits 是考试阅卷时的原始得分：

| 概念          | 类比                             |
| :---------- | :----------------------------- |
| **Logits**  | 阅卷老师打的原始分（可能满分 100，也可能超纲给 120） |
| **Softmax** | 把原始分换算成百分制排名                   |
| **采样策略**    | 校长决定怎么发录取通知书——只看第一名？还是前 3 名抽签？ |


## 代码视角

In [4]:
import torch
import torch.nn.functional as F

# 模型输出的 logits（5 个词）
logits = torch.tensor([3.2, 4.8, -1.5, 1.0, -10.0])

# 1. 转概率
probs = F.softmax(logits, dim=-1)
# tensor([0.504, 0.338, 0.005, 0.056, 0.000])

# 2. 贪心解码
next_token = torch.argmax(logits)  # → 0 (火锅)

# 3. 温度采样 (T=0.8)
temperature = 0.8
scaled_logits = logits / temperature
print(scaled_logits)
sampled = torch.multinomial(F.softmax(scaled_logits, dim=-1), 1)
print(sampled.item())

tensor([  4.0000,   6.0000,  -1.8750,   1.2500, -12.5000])
1


## 一句话总结
- Logits 是模型对"每个候选词有多合适"的原始打分——它本身不是概率，但它是所有采样策略的起点。分数越高越被看好， softmax 后变成概率，再通过各种策略选出下一个词。